Задание 2:

В PySpark приложении датафреймами(pyspark.sql.DataFrame) заданы продукты, категории и их связи. Каждому продукту может соответствовать несколько категорий или ни одной. А каждой категории может соответствовать несколько продуктов или ни одного. 

Напишите метод на PySpark, который в одном датафрейме вернет все пары «Имя продукта – Имя категории» и имена всех продуктов, у которых нет категорий.

In [1]:
from pyspark.sql import DataFrame
from pyspark.sql.functions import col

def get_product_category_pairs(products_df: DataFrame, categories_df: DataFrame, product_category_links_df: DataFrame) -> DataFrame:
    """
    Возвращает датафрейм со всеми парами "Имя продукта - Имя категории" 
    и продуктами без категорий.
    
    :products_df: Датафрейм продуктов с колонками ['product_id', 'product_name']
    :categories_df: Датафрейм категорий с колонками ['category_id', 'category_name']
    :product_category_links_df: Датафрейм связей с колонками ['product_id', 'category_id']
    
    :return:
        Датафрейм с колонками ['product_name', 'category_name'], содержащий:
        - все пары продукт-категория
        - продукты без категорий (с NULL в category_name)
    """
    products_with_categories = products_df \
        .join(product_category_links_df, on='product_id', how='left') \
        .join(categories_df, on='category_id', how='left') \
        .select(col('product_name'), col('category_name'))
    
    return products_with_categories

In [4]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

spark = SparkSession.builder.appName("ProductCategoryExample").getOrCreate()

# Схемы данных
product_schema = StructType([
    StructField("product_id", IntegerType(), False),
    StructField("product_name", StringType(), False)
])

category_schema = StructType([
    StructField("category_id", IntegerType(), False),
    StructField("category_name", StringType(), False)
])

link_schema = StructType([
    StructField("product_id", IntegerType(), False),
    StructField("category_id", IntegerType(), False)
])

# Пример данных
products_data = [(1, "Ноутбук"), (2, "Молоко"), (3, "Наушники"), (4, "Книга")]
categories_data = [(1, "Электроника"), (2, "Продукты"), (3, "Техника")]
links_data = [(1, 1), (2, 2), (3, 3), (1, 3)]

# Создаем датафреймы
products_df = spark.createDataFrame(products_data, product_schema)
categories_df = spark.createDataFrame(categories_data, category_schema)
product_category_links_df = spark.createDataFrame(links_data, link_schema)

# Вызываем наш метод
result = get_product_category_pairs(products_df, categories_df, product_category_links_df)
result.show()

+------------+-------------+
|product_name|category_name|
+------------+-------------+
|       Книга|         NULL|
|     Ноутбук|  Электроника|
|     Ноутбук|      Техника|
|    Наушники|      Техника|
|      Молоко|     Продукты|
+------------+-------------+

